# Multimodal Embedding Pipeline

Demonstrates **Qwen3-VL-Embedding** across all input types:
- Text
- PDF (rendered page-by-page)
- Slides (PPTX → PDF → images via LibreOffice)
- Video (keyframe extraction at 1fps)

All non-text types are converted to images before embedding.
Embeddings stored in ChromaDB with cosine similarity for semantic search.

> Model and quantization settings are configurable in the **Core functions** cell.

Runtime: GPU (T4 or better recommended).

In [ ]:
!pip install -q "chromadb==0.5.23" sentence-transformers "transformers>=4.57.0" qwen-vl-utils pymupdf python-pptx opencv-python Pillow bitsandbytes
!apt-get install -q libreoffice

## Core functions

In [ ]:
from PIL import Image
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig
import torch

# Switch between "Qwen/Qwen3-VL-Embedding-2B" and "Qwen/Qwen3-VL-Embedding-8B"
MODEL_NAME = "Qwen/Qwen3-VL-Embedding-8B"
USE_4BIT = True  # set False if running on a GPU with enough VRAM

_model = None

def _get_model():
    global _model
    if _model is None:
        kwargs = {}
        if USE_4BIT:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        _model = SentenceTransformer(MODEL_NAME, model_kwargs=kwargs)
    return _model


def embed_texts(texts, instruction="Represent the user's input."):
    return _get_model().encode(texts, prompt=instruction, convert_to_numpy=True).tolist()

def embed_query(query):
    return _get_model().encode(
        [query],
        prompt="Retrieve relevant documents for the query.",
        convert_to_numpy=True,
    )[0].tolist()

def embed_images(images, batch_size=1):
    model = _get_model()
    all_embeddings = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        embeddings = model.encode(batch, convert_to_numpy=True)
        all_embeddings.extend(embeddings.tolist())
    return all_embeddings

def embed_multimodal(items, batch_size=1):
    """items: list of str, PIL Image, or {"text": ..., "image": ...} dicts"""
    model = _get_model()
    all_embeddings = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        embeddings = model.encode(batch, convert_to_numpy=True)
        all_embeddings.extend(embeddings.tolist())
    return all_embeddings

In [ ]:
from pathlib import Path

MAX_IMAGE_PX = 512

def _resize(img, max_px=MAX_IMAGE_PX):
    w, h = img.size
    if max(w, h) <= max_px:
        return img
    scale = max_px / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

def pdf_to_images(pdf_path):
    import fitz
    doc = fitz.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=96)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(_resize(img))
    doc.close()
    return images

def pptx_to_images(pptx_path, output_dir="/tmp/slides"):
    import subprocess
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["libreoffice", "--headless", "--convert-to", "pdf", "--outdir", str(out), pptx_path],
        check=True,
    )
    pdf_path = out / (Path(pptx_path).stem + ".pdf")
    return pdf_to_images(str(pdf_path))

def video_to_keyframes(video_path, fps=1.0, max_frames=64):
    import cv2
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, int(video_fps / fps))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(_resize(Image.fromarray(rgb)))
        if len(frames) >= max_frames:
            break
    cap.release()
    return frames


In [ ]:
import chromadb
from datetime import datetime, timezone

_client = chromadb.Client()

COLLECTION_NAME = f"arena_embeddings_{MODEL_NAME.split('-')[-1].lower()}"  # arena_embeddings_8b or arena_embeddings_2b

collection = _client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)


def store_embeddings(labels, embeddings, metadata=None):
    ts = datetime.now(timezone.utc).isoformat()
    ids = [f"doc_{ts}_{i}" for i in range(len(labels))]
    meta = metadata or [{"timestamp": ts} for _ in labels]
    collection.add(ids=ids, embeddings=embeddings, documents=labels, metadatas=meta)
    return ids

def semantic_search(query_text, n_results=5):
    results = collection.query(
        query_embeddings=[embed_query(query_text)],
        n_results=n_results,
        include=["documents", "distances", "metadatas"],
    )
    print(f"Query: '{query_text}'\n")
    for i, (doc, dist, meta) in enumerate(zip(results["documents"][0], results["distances"][0], results["metadatas"][0])):
        print(f"{i+1}. [score: {1 - dist:.4f}] [{meta.get('source', '?')}] {doc}")

print('ChromaDB ready.')

KeyboardInterrupt: 

## 1. Load model

In [ ]:
print(f"Loading {MODEL_NAME} ({'4-bit quantized' if USE_4BIT else 'full precision'})...")
_get_model()
print("✓ Model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Model ready.


## 2. Text

In [ ]:
texts = [
    "B2B SaaS email campaign promoting enterprise plan to CTOs.",
    "Summer Instagram sale targeting young adults with 20% discount.",
    "Facebook retargeting campaign for abandoned shopping cart users.",
]
text_embeddings = embed_texts(texts)
store_embeddings(texts, text_embeddings, [{"source": "text"} for _ in texts])
print(f"✓ Text — {len(text_embeddings)} embeddings stored (dim: {len(text_embeddings[0])})")

Text — 3 embeddings, dim: 2048


## 3. PDF

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload one or more PDFs

all_pdf_images = []
all_pdf_labels = []
all_pdf_meta = []

for pdf_path in uploaded.keys():
    images = pdf_to_images(pdf_path)
    print(f"  {pdf_path} — {len(images)} pages")
    all_pdf_images.extend(images)
    all_pdf_labels.extend([f"{pdf_path} — page {i+1}" for i in range(len(images))])
    all_pdf_meta.extend([{"source": "pdf", "file": pdf_path, "page": i+1} for i in range(len(images))])

pdf_embeddings = embed_images(all_pdf_images)
store_embeddings(all_pdf_labels, pdf_embeddings, all_pdf_meta)
print(f"\n✓ PDF — {len(pdf_embeddings)} embeddings stored ({len(all_pdf_images)} pages across {len(uploaded)} file(s))")
print(f"  Total in DB: {collection.count()}")

Saving Sleep_Deprivation_Research_Summary.pdf to Sleep_Deprivation_Research_Summary (3).pdf
Saving saas_marketing_report.pdf to saas_marketing_report (3).pdf
Sleep_Deprivation_Research_Summary (3).pdf — 3 pages
saas_marketing_report (3).pdf — 3 pages

PDF total — 6 embeddings stored


## 4. Slides (PPTX)

In [ ]:
uploaded = files.upload()  # upload one or more PPTX files

all_slide_images = []
all_slide_labels = []
all_slide_meta = []

for pptx_path in uploaded.keys():
    images = pptx_to_images(pptx_path)
    print(f"  {pptx_path} — {len(images)} slides")
    all_slide_images.extend(images)
    all_slide_labels.extend([f"{pptx_path} — slide {i+1}" for i in range(len(images))])
    all_slide_meta.extend([{"source": "pptx", "file": pptx_path, "slide": i+1} for i in range(len(images))])

slide_embeddings = embed_images(all_slide_images)
store_embeddings(all_slide_labels, slide_embeddings, all_slide_meta)
print(f"\n✓ PPTX — {len(slide_embeddings)} embeddings stored ({len(all_slide_images)} slides across {len(uploaded)} file(s))")
print(f"  Total in DB: {collection.count()}")

Saving Oceans in Peril_ Climate Change Impacts.pptx to Oceans in Peril_ Climate Change Impacts.pptx
Saving Nike Air Max Ignite Launch Strategy.pptx to Nike Air Max Ignite Launch Strategy.pptx
Oceans in Peril_ Climate Change Impacts.pptx — 6 slides
Nike Air Max Ignite Launch Strategy.pptx — 6 slides

PPTX total — 12 embeddings stored


## 5. Video

In [ ]:
uploaded = files.upload()  # upload one or more videos

all_frames = []
all_frame_labels = []
all_frame_meta = []

for video_path in uploaded.keys():
    frames = video_to_keyframes(video_path, fps=1.0)
    print(f"  {video_path} — {len(frames)} keyframes extracted")
    all_frames.extend(frames)
    all_frame_labels.extend([f"{video_path} — frame {i+1}" for i in range(len(frames))])
    all_frame_meta.extend([{"source": "video", "file": video_path, "frame": i+1} for i in range(len(frames))])

video_embeddings = embed_images(all_frames)
store_embeddings(all_frame_labels, video_embeddings, all_frame_meta)
print(f"\n✓ Video — {len(video_embeddings)} embeddings stored ({len(all_frames)} frames across {len(uploaded)} file(s))")
print(f"  Total in DB: {collection.count()}")

Saving 8533450-hd_1280_720_25fps.mp4 to 8533450-hd_1280_720_25fps.mp4
Video — 39 keyframes extracted
Video — 39 embeddings stored


## 6. Semantic Search

Query across all stored content — text, PDF, PPTX, and video frames.

In [ ]:
print(f"Total stored: {collection.count()} items\n")
print("=" * 60)

semantic_search("marketing campaign for enterprise software", 5)
print()
semantic_search("Nike product launch strategy", 5)
print()
semantic_search("effects of sleep deprivation on cognitive performance", 5)

Total stored: 60 items

Query: 'marketing campaign for nike'

1. [score: 0.5604] [pptx] Nike Air Max Ignite Launch Strategy.pptx — slide 3
2. [score: 0.5082] [pptx] Nike Air Max Ignite Launch Strategy.pptx — slide 4
3. [score: 0.4712] [pptx] Nike Air Max Ignite Launch Strategy.pptx — slide 2
4. [score: 0.4453] [pptx] Nike Air Max Ignite Launch Strategy.pptx — slide 1
5. [score: 0.4372] [text] Facebook retargeting campaign for abandoned shopping cart users.
6. [score: 0.4142] [video] 8533450-hd_1280_720_25fps.mp4 — frame 6
7. [score: 0.3923] [text] B2B SaaS email campaign promoting enterprise plan to CTOs.
8. [score: 0.3825] [text] Summer Instagram sale targeting young adults with 20% discount.
9. [score: 0.3781] [video] 8533450-hd_1280_720_25fps.mp4 — frame 12
10. [score: 0.3687] [video] 8533450-hd_1280_720_25fps.mp4 — frame 11
